<a href="https://colab.research.google.com/github/asopozala-prog/google_colab_ML/blob/main/imdb_sentiment_lstm_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IMDB Sentiment Analysis – BiLSTM with Attention

This notebook extends the previous `imdb_sentiment_lstm` model by adding a
self-attention mechanism on top of a bidirectional LSTM encoder.

Goal:
- Improve the model's ability to focus on important words (e.g., negations such as "not")
- Better handle nuanced sentiment compared to a simple BiLSTM + pooling model.


In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.20.0
GPUs: []


In [2]:
# Load IMDB reviews (text + label)
train_raw, test_raw = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True,  # (text, label)
)

BUFFER_SIZE = 10_000
BATCH_SIZE = 64

# Shuffle training data
train_raw = train_raw.shuffle(
    BUFFER_SIZE, reshuffle_each_iteration=True
)

# Validation split from training data
VAL_SIZE = 5_000
val_raw = train_raw.take(VAL_SIZE)
train_raw = train_raw.skip(VAL_SIZE)

len(list(val_raw)), "validation examples approx"


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.VFPXI3_1.0.0/imdb_reviews-train.tfrecor…

...

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.VFPXI3_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.VFPXI3_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


(5000, 'validation examples approx')

In [3]:
VOCAB_SIZE = 20_000
SEQUENCE_LENGTH = 200

vectorize_layer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
)

# Adapt on raw training text only (no labels)
train_text_only = train_raw.map(lambda x, y: x)
vectorize_layer.adapt(train_text_only)

def vectorize_text(text, label):
    text = tf.expand_dims(text, -1)  # shape adjustment
    return vectorize_layer(text), label

# Build final datasets: vectorized + batched + prefetched
train_ds = (
    train_raw
    .batch(BATCH_SIZE)
    .map(vectorize_text)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_raw
    .batch(BATCH_SIZE)
    .map(vectorize_text)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_raw
    .batch(BATCH_SIZE)
    .map(vectorize_text)
    .prefetch(tf.data.AUTOTUNE)
)

for batch_x, batch_y in train_ds.take(1):
    print("Vectorized batch shape:", batch_x.shape)
    print("Labels batch shape:", batch_y.shape)


Vectorized batch shape: (64, 200)
Labels batch shape: (64,)


In [4]:
EMBEDDING_DIM = 128

inputs = layers.Input(shape=(SEQUENCE_LENGTH,), dtype="int64")

# 1) Embedding
x = layers.Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
    name="word_embedding",
)(inputs)

# 2) Bidirectional LSTM returning full sequence
x = layers.Bidirectional(
    layers.LSTM(64, return_sequences=True)
)(x)  # shape: (batch, seq_len, 128)

# 3) Self-attention: queries=keys=values = LSTM outputs
attended = layers.Attention(name="self_attention")([x, x])
# shape: (batch, seq_len, 128)

# 4) Pool across time (average over sequence positions)
x = layers.GlobalAveragePooling1D()(attended)

# 5) Classification head
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs, name="imdb_sentiment_lstm_attention")
model.summary()


Model: "imdb_sentiment_lstm_attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_embedding      │ (None, 200, 128)  │  2,560,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 200, 128)  │     98,816 │ word_embedding[0… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attention      │ (None, 200, 128)  │          0 │ bidirectional[0]… │
│ (Attention)         │                   │            │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ self_attention[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      8,256 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,667,137 (10.17 MB)

 Trainable params: 2,667,137 (10.17 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

EPOCHS = 5  # you can reduce to 3 if training is too slow

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)


Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 233s 733ms/step - accuracy: 0.7742 - loss: 0.4543 - val_accuracy: 0.8786 - val_loss: 0.3087
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 229s 730ms/step - accuracy: 0.9100 - loss: 0.2384 - val_accuracy: 0.9236 - val_loss: 0.2103
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 233s 743ms/step - accuracy: 0.9423 - loss: 0.1596 - val_accuracy: 0.9508 - val_loss: 0.1351
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 258s 729ms/step - accuracy: 0.9632 - loss: 0.1095 - val_accuracy: 0.9546 - val_loss: 0.1286
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 227s 724ms/step - accuracy: 0.9743 - loss: 0.0757 - val_accuracy: 0.9796 - val_loss: 0.0671


In [6]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test loss:", test_loss)
print("Test accuracy:", test_acc)


391/391 ━━━━━━━━━━━━━━━━━━━━ 83s 213ms/step - accuracy: 0.8360 - loss: 0.5214
Test loss: 0.5214300155639648
Test accuracy: 0.8360000252723694


In [7]:
def predict_review(text: str, threshold: float = 0.5):
    text_tensor = tf.constant([text])
    vectorized = vectorize_layer(text_tensor)
    prob = model.predict(vectorized)[0][0]
    label = "positive" if prob >= threshold else "negative"
    return label, float(prob)

examples = [
    "An amiable, warm-hearted, inoffensive family animated film about a dog who stars in a popular television show.",
    "Bolt is certainly not one of Disney's greatest hits.",
    "I really wanted to like this movie, but it just never clicked for me.",
    "It's not terrible, but it's definitely not good either.",
]

for review in examples:
    label, p = predict_review(review)
    print("Review:", review)
    print(f"Prediction: {label} (p={p:.3f})\n")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 370ms/step
Review: An amiable, warm-hearted, inoffensive family animated film about a dog who stars in a popular television show.
Prediction: positive (p=0.548)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Review: Bolt is certainly not one of Disney's greatest hits.
Prediction: positive (p=0.558)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Review: I really wanted to like this movie, but it just never clicked for me.
Prediction: positive (p=0.723)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Review: It's not terrible, but it's definitely not good either.
Prediction: positive (p=0.513)

